# TD 3 — NLP (BoW & TF-IDF)

Dans ce notebook, on va faire les 3 exercices du TD :

1. Construire une matrice **Bag of Words (BoW)** à partir de 3 phrases.
2. Calculer des scores **TF-IDF** sur 3 documents et donner la valeur TF-IDF du terme **"revolutionizing"**.
3. Calculer des scores **TF-IDF** sur 3 documents et donner la valeur TF-IDF du terme **"AI"**.

On utilise une tokenisation **simple** :
- minuscules
- suppression de la ponctuation
- découpage par espaces


# Imports + fonctions utilitaires

In [1]:
import re
import numpy as np
import pandas as pd

def tokenize(text):
    """
    Tokenisation simple :
    - minuscule
    - suppression ponctuation
    - split par espaces
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)  # garde lettres/chiffres/espaces
    tokens = text.split()
    return tokens

def build_vocab(tokenized_texts):
    """
    Construit le vocabulaire (liste triée) à partir d'une liste de listes de tokens.
    """
    vocab = sorted(set(token for doc in tokenized_texts for token in doc))
    return vocab

def bow_matrix(tokenized_texts, vocab):
    """
    Construit la matrice BoW (comptage) : lignes = documents, colonnes = mots.
    """
    mat = np.zeros((len(tokenized_texts), len(vocab)), dtype=int)
    word_to_idx = {w:i for i,w in enumerate(vocab)}
    for i, doc in enumerate(tokenized_texts):
        for w in doc:
            mat[i, word_to_idx[w]] += 1
    return mat

def df_counts(tokenized_texts, vocab):
    """
    df(w) = nombre de documents contenant w (au moins 1 fois)
    """
    word_to_idx = {w:i for i,w in enumerate(vocab)}
    df = np.zeros(len(vocab), dtype=int)
    for doc in tokenized_texts:
        unique_words = set(doc)
        for w in unique_words:
            df[word_to_idx[w]] += 1
    return df

def idf_scores(df, N):
    """
    idf(w) = log10(N/df(w))
    """
    return np.log10(N / df)

def tfidf_matrix(tf_mat, idf_vec):
    """
    TF-IDF = TF * IDF (sans normalisation)
    """
    return tf_mat * idf_vec


## Exercice 1 : Bag of Words (BoW)

Phrases :
1. "Machine learning is fun."
2. "Learning Python is interesting."
3. "Machine learning and Python are related."

Objectif :
1. Tokeniser chaque phrase (minuscule + suppression ponctuation).
2. Construire le vocabulaire global (tous les mots uniques).
3. Construire la matrice BoW : 
   - lignes = phrases
   - colonnes = mots du vocabulaire
   - valeurs = nombre d'occurrences du mot dans la phrase


Cellule Code — BoW

In [2]:
phrases = [
    "Machine learning is fun.",
    "Learning Python is interesting.",
    "Machine learning and Python are related."
]

tokens_phrases = [tokenize(p) for p in phrases]
vocab1 = build_vocab(tokens_phrases)

bow1 = bow_matrix(tokens_phrases, vocab1)

df_bow1 = pd.DataFrame(bow1, columns=vocab1, index=[f"Phrase {i+1}" for i in range(len(phrases))])
df_bow1


,and,are,fun,interesting,is,learning,machine,python,related
Phrase 1,0,0,1,0,1,1,1,0,0
Phrase 2,0,0,0,1,1,1,0,1,0
Phrase 3,1,1,0,0,0,1,1,1,1


## Interprétation — Exercice 1 (Bag of Words)

La matrice Bag of Words construite représente la fréquence d’apparition de chaque mot
du vocabulaire dans chacune des phrases.

- Les **lignes** correspondent aux phrases (Phrase 1, Phrase 2, Phrase 3).
- Les **colonnes** correspondent aux mots du vocabulaire global extrait de l’ensemble
  des phrases : `and, are, fun, interesting, is, learning, machine, python, related`.
- Chaque cellule contient le **nombre d’occurrences** du mot dans la phrase correspondante.

Observations :
- Les mots **"machine"** et **"learning"** apparaissent dans les phrases 1 et 3,
  ce qui montre une similarité thématique entre ces deux phrases.
- Le mot **"python"** apparaît dans les phrases 2 et 3.
- Les mots **"fun"** et **"interesting"** sont spécifiques respectivement à la
  phrase 1 et à la phrase 2.
- Les mots **"and"**, **"are"** et **"related"** apparaissent uniquement dans la
  phrase 3, ce qui la rend plus riche en vocabulaire.


## Conclusion — Exercice 1

La représentation Bag of Words permet de transformer des phrases textuelles
en vecteurs numériques simples basés uniquement sur la fréquence des mots.

Cependant :
- L’ordre des mots est perdu.
- Le sens et le contexte ne sont pas pris en compte.
- Tous les mots ont la même importance, même les mots très fréquents comme "is".

Cette représentation reste néanmoins une base fondamentale en NLP et servira
de point de départ pour des méthodes plus avancées comme le **TF-IDF**,
qui pondère les mots selon leur importance dans les documents.


## Exercice 2 : TF-IDF

Documents :
- Doc1 : "AI is revolutionizing industries."
- Doc2 : "AI and machine learning are revolutionizing."
- Doc3 : "Machine learning is a part of AI."

Objectif :
1. Tokeniser les documents.
2. Construire vocabulaire global.
3. Construire matrice TF (comptage).
4. Calculer df et idf :
   - df(w) = nb de documents contenant w
   - idf(w) = log10(N/df(w)), avec N=3
5. Calculer TF-IDF = TF * IDF (sans normalisation).
6. Donner TF-IDF du terme "revolutionizing" (par document).


Cellule Code — Calcul TF-IDF

In [4]:
docs2 = [
    "AI is revolutionizing industries.",
    "AI and machine learning are revolutionizing.",
    "Machine learning is a part of AI."
]

tokens_docs2 = [tokenize(d) for d in docs2]
vocab2 = build_vocab(tokens_docs2)

tf2 = bow_matrix(tokens_docs2, vocab2)  # TF = BoW (comptage)
df2 = df_counts(tokens_docs2, vocab2)
idf2 = idf_scores(df2, N=len(docs2))

tfidf2 = tfidf_matrix(tf2, idf2)

df_tfidf2 = pd.DataFrame(tfidf2, columns=vocab2, index=[f"Doc {i+1}" for i in range(len(docs2))])
df_tfidf2


,a,ai,and,are,industries,is,learning,machine,of,part,revolutionizing
Doc 1,0.000000,0.0,0.000000,0.000000,0.477121,0.176091,0.000000,0.000000,0.000000,0.000000,0.176091
Doc 2,0.000000,0.0,0.477121,0.477121,0.000000,0.000000,0.176091,0.176091,0.000000,0.000000,0.176091
Doc 3,0.477121,0.0,0.000000,0.000000,0.000000,0.176091,0.176091,0.176091,0.477121,0.477121,0.000000


Cellule Code — Valeur exacte pour “revolutionizing

In [5]:
term = "revolutionizing"
idx = vocab2.index(term)

print("Terme :", term)
print("df(revolutionizing) =", df2[idx])
print("idf(revolutionizing) =", idf2[idx])

for i in range(len(docs2)):
    print(f"TF-IDF({term}, Doc{i+1}) =", tfidf2[i, idx])


Terme : revolutionizing
df(revolutionizing) = 2
idf(revolutionizing) = 0.17609125905568124
TF-IDF(revolutionizing, Doc1) = 0.17609125905568124
TF-IDF(revolutionizing, Doc2) = 0.17609125905568124
TF-IDF(revolutionizing, Doc3) = 0.0


## Interprétation — Exercice 2 (TF-IDF)

On a calculé la matrice **TF-IDF** pour les 3 documents.

Rappel :
- **TF (Term Frequency)** : nombre d’occurrences du mot dans un document.
- **df (Document Frequency)** : nombre de documents contenant le mot.
- **IDF (Inverse Document Frequency)** : idf(w) = log10(N / df(w)) avec N = 3.
- **TF-IDF** = TF × IDF (sans normalisation).

### Analyse du terme "revolutionizing"
Le mot **"revolutionizing"** apparaît dans :
- Doc1 : oui
- Doc2 : oui
- Doc3 : non

Donc :
- df("revolutionizing") = 2
- idf("revolutionizing") = log10(3/2) ≈ 0.17609

Comme TF("revolutionizing") = 1 dans Doc1 et Doc2, on obtient :
- TF-IDF("revolutionizing", Doc1) = 1 × 0.17609 = 0.17609
- TF-IDF("revolutionizing", Doc2) = 1 × 0.17609 = 0.17609
- TF-IDF("revolutionizing", Doc3) = 0 (le mot n’apparaît pas)


## Conclusion — Exercice 2

Le score TF-IDF du mot **"revolutionizing"** est identique dans Doc1 et Doc2 car :
- le terme apparaît exactement une fois dans chacun de ces documents (TF=1),
- et son IDF est le même car df=2.

Le document 3 obtient un score nul pour ce terme car il ne contient pas le mot.

TF-IDF permet donc de **mettre en valeur les mots discriminants** :
- un mot présent dans peu de documents aura un IDF plus grand,
- un mot présent dans tous les documents aura un IDF proche de 0 et sera moins informatif.


## Exercice 3 : TF-IDF

Documents :
- Doc1 : "Deep learning revolutionizes AI."
- Doc2 : "Machine learning is a subset of AI."
- Doc3 : "AI and deep learning are related."

Objectif :
Même méthode que l’exercice 2, mais on veut la valeur TF-IDF pour le terme "AI".


Cellule Code — Calcul TF-IDF

In [6]:
docs3 = [
    "Deep learning revolutionizes AI.",
    "Machine learning is a subset of AI.",
    "AI and deep learning are related."
]

tokens_docs3 = [tokenize(d) for d in docs3]
vocab3 = build_vocab(tokens_docs3)

tf3 = bow_matrix(tokens_docs3, vocab3)
df3 = df_counts(tokens_docs3, vocab3)
idf3 = idf_scores(df3, N=len(docs3))

tfidf3 = tfidf_matrix(tf3, idf3)

df_tfidf3 = pd.DataFrame(tfidf3, columns=vocab3, index=[f"Doc {i+1}" for i in range(len(docs3))])
df_tfidf3


,a,ai,and,are,deep,is,learning,machine,of,related,revolutionizes,subset
Doc 1,0.000000,0.0,0.000000,0.000000,0.176091,0.000000,0.0,0.000000,0.000000,0.000000,0.477121,0.000000
Doc 2,0.477121,0.0,0.000000,0.000000,0.000000,0.477121,0.0,0.477121,0.477121,0.000000,0.000000,0.477121
Doc 3,0.000000,0.0,0.477121,0.477121,0.176091,0.000000,0.0,0.000000,0.000000,0.477121,0.000000,0.000000


Cellule Code — Valeur exacte pour 'ai'

In [7]:
term = "ai"
idx = vocab3.index(term)

print("Terme :", term)
print("df(ai) =", df3[idx])
print("idf(ai) =", idf3[idx])

for i in range(len(docs3)):
    print(f"TF-IDF({term}, Doc{i+1}) =", tfidf3[i, idx])


Terme : ai
df(ai) = 3
idf(ai) = 0.0
TF-IDF(ai, Doc1) = 0.0
TF-IDF(ai, Doc2) = 0.0
TF-IDF(ai, Doc3) = 0.0


## Interprétation — Exercice 3 (TF-IDF)

On a calculé la matrice **TF-IDF** pour les 3 documents du dataset.

Documents :
- Doc1 : "Deep learning revolutionizes AI."
- Doc2 : "Machine learning is a subset of AI."
- Doc3 : "AI and deep learning are related."

### Analyse du terme "AI"

Le mot **"AI"** apparaît dans :
- Doc1
- Doc2
- Doc3

Donc :
- df("AI") = 3
- N = 3 documents
- idf("AI") = log10(3/3) = 0

Même si le terme **"AI"** apparaît dans chaque document (TF = 1),
son **IDF est nul**, ce qui entraîne :

- TF-IDF("AI", Doc1) = 0
- TF-IDF("AI", Doc2) = 0
- TF-IDF("AI", Doc3) = 0

Cela signifie que le terme "AI" n’apporte aucune information
discriminante entre les documents.


## Conclusion — Exercice 3

Le calcul TF-IDF montre que les termes apparaissant dans **tous les documents**
obtiennent un score nul, même s’ils sont fréquents.

Le mot **"AI"** est un terme très commun dans ce corpus, ce qui explique :
- un df maximal,
- un idf nul,
- et donc un TF-IDF nul dans tous les documents.

TF-IDF permet ainsi d’éliminer automatiquement les mots trop généraux
et de mettre en valeur les termes réellement discriminants du corpus,
comme "deep", "subset" ou "revolutionizes".


## Conclusion générale

Dans ce TD, nous avons :
1. Utilisé le **Bag of Words** pour représenter des phrases sous forme vectorielle.
2. Appliqué **TF-IDF** pour pondérer les mots selon leur importance dans le corpus.
3. Observé que :
   - les mots rares sont valorisés,
   - les mots très fréquents sont pénalisés,
   - TF-IDF est plus informatif que BoW pour comparer des documents.

Ces représentations constituent la base de nombreuses méthodes
en recherche d’information et en apprentissage automatique pour le NLP.
